# Automatidata project

## Course 5 - The Nuts and Bolts of Machine Learning

Automatidata is working with the **New York City Taxi and Limousine Commission (NYC TLC)**.

The client requested a machine learning model related to passenger tipping behavior. Because a model that alerts drivers about individual passengers could lead to unfair treatment, the objective is reframed as an internal analytical model that predicts whether a credit-card customer is a **generous tipper**, defined as leaving a tip of at least 20%.

The notebook follows the PACE framework:

- Plan
- Analyze
- Construct
- Execute


# Course 5 end-of-course project: Build a machine learning model

This project uses tree-based classification models to predict a binary target.

The activity has three parts:

1. Ethical considerations
2. Feature engineering
3. Modeling and evaluation


## PACE: Plan

### Ethical considerations

A real-time alert about individual passengers could encourage discrimination or unequal service.

- A false prediction can frustrate drivers and reduce trust in the system.
- A passenger incorrectly classified as unlikely to tip could receive slower or poorer service.
- The safer use is internal business intelligence and aggregate trend analysis rather than real-time passenger screening.

### Modified objective

Predict whether a credit-card passenger is particularly generous, meaning that the passenger leaves a tip of **20% or more**.

### Target variable

`generous`

- `1`: tip percentage is at least 20%
- `0`: tip percentage is below 20%

### Main evaluation metric

The **F1 score** balances precision and recall. This is appropriate because both false positives and false negatives have meaningful consequences.


### Task 1. Imports and data loading


In [ ]:
# Optional Colab setup: install XGBoost only if it is unavailable
try:
    import xgboost
except ImportError:
    import subprocess
    import sys
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "xgboost"]
    )


In [ ]:
# Import packages and libraries

# Data manipulation and visualization
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Model selection and evaluation
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    PrecisionRecallDisplay,
)

# Tree-based models
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier, plot_importance

# Saving and loading fitted models
import pickle


In [ ]:
# RUN THIS CELL TO SEE ALL COLUMNS
pd.set_option("display.max_columns", None)


### Colab-compatible data loading

The original lab uses two files:

- `2017_Yellow_Taxi_Trip_Data.csv`
- `nyc_preds_means.csv`

The loader searches for local copies in Google Colab and then tries public GitHub paths.

If `nyc_preds_means.csv` is unavailable, the notebook reconstructs:

- `mean_duration`
- `mean_distance`
- `predicted_fare`


In [ ]:
# RUN THE CELL BELOW TO IMPORT YOUR DATA

from pathlib import Path

TAXI_FILENAME = "2017_Yellow_Taxi_Trip_Data.csv"
PREDS_FILENAME = "nyc_preds_means.csv"

# Expected GitHub folder for this course
COURSE_5_FOLDER = "5.%20The%20Nuts%20and%20bolts%20of%20machine%20learning"

local_taxi_candidates = [
    Path(TAXI_FILENAME),
    Path("/content") / TAXI_FILENAME,
    Path("/mnt/data") / TAXI_FILENAME,
]

local_preds_candidates = [
    Path(PREDS_FILENAME),
    Path("/content") / PREDS_FILENAME,
    Path("/mnt/data") / PREDS_FILENAME,
]

taxi_urls = [
    (
        "https://raw.githubusercontent.com/"
        "Abraham123w/Google-Advanced-Data-Analytics-Professional-Certificate/"
        f"main/{COURSE_5_FOLDER}/{TAXI_FILENAME}"
    ),
    (
        "https://raw.githubusercontent.com/"
        "Abraham123w/Google-Advanced-Data-Analytics-Professional-Certificate/"
        "main/4.%20Regression%20Analysis%20Simplify%20Complex%20Data%20Relationships/"
        f"{TAXI_FILENAME}"
    ),
    (
        "https://raw.githubusercontent.com/"
        "Abraham123w/Google-Advanced-Data-Analytics-Professional-Certificate/"
        "main/3.%20The%20Power%20of%20Statistics/"
        f"{TAXI_FILENAME}"
    ),
]

preds_urls = [
    (
        "https://raw.githubusercontent.com/"
        "Abraham123w/Google-Advanced-Data-Analytics-Professional-Certificate/"
        f"main/{COURSE_5_FOLDER}/{PREDS_FILENAME}"
    ),
    (
        "https://raw.githubusercontent.com/"
        "Abraham123w/Google-Advanced-Data-Analytics-Professional-Certificate/"
        "main/4.%20Regression%20Analysis%20Simplify%20Complex%20Data%20Relationships/"
        f"{PREDS_FILENAME}"
    ),
]


def read_csv_with_fallback(local_candidates, urls, label):
    errors = []

    for candidate in local_candidates:
        try:
            if candidate.exists() and candidate.stat().st_size > 100:
                frame = pd.read_csv(candidate)
                if not frame.empty:
                    print(f"{label} cargado desde: {candidate}")
                    return frame
        except Exception as error:
            errors.append(f"{candidate}: {error}")

    for url in urls:
        try:
            frame = pd.read_csv(url)
            if not frame.empty:
                print(f"{label} cargado desde: {url}")
                return frame
        except Exception as error:
            errors.append(f"{url}: {error}")

    print(f"No se encontró {label}.")
    for error in errors:
        print("-", error)

    return None


df0 = read_csv_with_fallback(
    local_candidates=local_taxi_candidates,
    urls=taxi_urls,
    label=TAXI_FILENAME,
)

if df0 is None:
    raise RuntimeError(
        "No fue posible cargar 2017_Yellow_Taxi_Trip_Data.csv. "
        "Comprueba que el repositorio sea público y el nombre sea exacto."
    )


def build_preds_means_fallback(taxi_df):
    from sklearn.linear_model import LinearRegression
    from sklearn.preprocessing import StandardScaler
    from sklearn.model_selection import train_test_split

    work = taxi_df.copy()

    work["tpep_pickup_datetime"] = pd.to_datetime(
        work["tpep_pickup_datetime"]
    )
    work["tpep_dropoff_datetime"] = pd.to_datetime(
        work["tpep_dropoff_datetime"]
    )

    work["duration"] = (
        work["tpep_dropoff_datetime"]
        - work["tpep_pickup_datetime"]
    ).dt.total_seconds() / 60

    work.loc[work["fare_amount"] < 0, "fare_amount"] = 0
    work.loc[work["duration"] < 0, "duration"] = 0

    for column in ["fare_amount", "duration"]:
        q1 = work[column].quantile(0.25)
        q3 = work[column].quantile(0.75)
        iqr = q3 - q1
        upper_threshold = q3 + (6 * iqr)
        work.loc[work[column] > upper_threshold, column] = upper_threshold

    work["pickup_dropoff"] = (
        work["PULocationID"].astype(str)
        + " "
        + work["DOLocationID"].astype(str)
    )

    route_mean_distance = work.groupby(
        "pickup_dropoff"
    )["trip_distance"].mean()

    route_mean_duration = work.groupby(
        "pickup_dropoff"
    )["duration"].mean()

    work["mean_distance"] = work["pickup_dropoff"].map(
        route_mean_distance
    )

    work["mean_duration"] = work["pickup_dropoff"].map(
        route_mean_duration
    )

    work["day"] = (
        work["tpep_pickup_datetime"]
        .dt.day_name()
        .str.lower()
    )

    work["month"] = (
        work["tpep_pickup_datetime"]
        .dt.month_name()
        .str.lower()
    )

    hour = work["tpep_pickup_datetime"].dt.hour

    work["rush_hour"] = (
        ((hour >= 6) & (hour < 10))
        | ((hour >= 16) & (hour < 20))
    ).astype(int)

    work.loc[
        work["day"].isin(["saturday", "sunday"]),
        "rush_hour",
    ] = 0

    model_frame = work[
        [
            "VendorID",
            "passenger_count",
            "mean_distance",
            "mean_duration",
            "day",
            "month",
            "rush_hour",
            "fare_amount",
        ]
    ].copy()

    X_fare = model_frame.drop(columns="fare_amount")
    y_fare = model_frame["fare_amount"]

    X_fare["VendorID"] = X_fare["VendorID"].astype(str)
    X_fare = pd.get_dummies(
        X_fare,
        drop_first=True,
        dtype=int,
    )

    X_train_fare, _, y_train_fare, _ = train_test_split(
        X_fare,
        y_fare,
        test_size=0.2,
        random_state=0,
    )

    scaler = StandardScaler()
    X_train_fare_scaled = scaler.fit_transform(X_train_fare)

    fare_model = LinearRegression()
    fare_model.fit(X_train_fare_scaled, y_train_fare)

    predicted_fare = fare_model.predict(
        scaler.transform(X_fare)
    )

    return pd.DataFrame(
        {
            "mean_duration": work["mean_duration"].to_numpy(),
            "mean_distance": work["mean_distance"].to_numpy(),
            "predicted_fare": predicted_fare,
        },
        index=taxi_df.index,
    )


nyc_preds_means = read_csv_with_fallback(
    local_candidates=local_preds_candidates,
    urls=preds_urls,
    label=PREDS_FILENAME,
)

if nyc_preds_means is None:
    print(
        "Se generará nyc_preds_means a partir del dataset original "
        "para que el notebook pueda ejecutarse en Colab."
    )
    nyc_preds_means = build_preds_means_fallback(df0)

print(f"df0: {df0.shape[0]:,} filas y {df0.shape[1]} columnas")
print(
    "nyc_preds_means: "
    f"{nyc_preds_means.shape[0]:,} filas y "
    f"{nyc_preds_means.shape[1]} columnas"
)


Inspect the first few rows of `df0`.


In [ ]:
# Inspect the first few rows of df0
df0.head(10)


Inspect the first few rows of `nyc_preds_means`.


In [ ]:
# Inspect the first few rows of nyc_preds_means
nyc_preds_means.head(10)


### Join the two dataframes


In [ ]:
# Merge datasets using their indexes
df0 = df0.merge(
    nyc_preds_means,
    left_index=True,
    right_index=True,
)

df0.head()


## PACE: Analyze

### Task 2. Feature engineering


In [ ]:
df0.info()


Customers who pay cash usually have a recorded tip amount of zero. The model therefore uses only credit-card trips.


In [ ]:
# Subset the data to isolate only customers who paid by credit card
df1 = df0.copy()
df1 = df1[df1["payment_type"] == 1]


### Create the target

Calculate tip percentage as:

`tip_amount / (total_amount - tip_amount)`

The result is rounded to three decimal places before applying the 20% threshold.


In [ ]:
# Floating-point arithmetic example
1.1 + 2.2


In [ ]:
# Create tip percentage column
df1["tip_percent"] = (
    df1["tip_amount"]
    / (df1["total_amount"] - df1["tip_amount"])
).round(3)


In [ ]:
# Create generous target
df1["generous"] = df1["tip_percent"]
df1["generous"] = df1["generous"] >= 0.20
df1["generous"] = df1["generous"].astype(int)


### Create day and time-of-day variables


In [ ]:
# Convert pickup and drop-off columns to datetime
df1["tpep_pickup_datetime"] = pd.to_datetime(
    df1["tpep_pickup_datetime"]
)
df1["tpep_dropoff_datetime"] = pd.to_datetime(
    df1["tpep_dropoff_datetime"]
)


In [ ]:
# Create day column
df1["day"] = (
    df1["tpep_pickup_datetime"]
    .dt.day_name()
    .str.lower()
)


In [ ]:
# Extract pickup hour and create four temporary columns
pickup_hour = df1["tpep_pickup_datetime"].dt.hour

df1["am_rush"] = pickup_hour
df1["daytime"] = pickup_hour
df1["pm_rush"] = pickup_hour
df1["nighttime"] = pickup_hour


In [ ]:
# Define am_rush conversion function [06:00-10:00)
def am_rush(hour):
    if 6 <= hour < 10:
        val = 1
    else:
        val = 0
    return val


In [ ]:
# Apply am_rush function
df1["am_rush"] = df1["am_rush"].apply(am_rush)
df1["am_rush"].head()


In [ ]:
# Define daytime conversion function [10:00-16:00)
def daytime(hour):
    if 10 <= hour < 16:
        val = 1
    else:
        val = 0
    return val


In [ ]:
# Apply daytime function
df1["daytime"] = df1["daytime"].apply(daytime)
df1["daytime"].head()


In [ ]:
# Define pm_rush conversion function [16:00-20:00)
def pm_rush(hour):
    if 16 <= hour < 20:
        val = 1
    else:
        val = 0
    return val


In [ ]:
# Apply pm_rush function
df1["pm_rush"] = df1["pm_rush"].apply(pm_rush)
df1["pm_rush"].head()


In [ ]:
# Define nighttime conversion function [20:00-06:00)
def nighttime(hour):
    if hour >= 20 or hour < 6:
        val = 1
    else:
        val = 0
    return val


In [ ]:
# Apply nighttime function
df1["nighttime"] = df1["nighttime"].apply(nighttime)
df1["nighttime"].head()


### Create month column


In [ ]:
# Create abbreviated lowercase month column
df1["month"] = (
    df1["tpep_pickup_datetime"]
    .dt.strftime("%b")
    .str.lower()
)


In [ ]:
# Examine first rows
df1.head()


### Drop redundant, irrelevant, and leakage columns

The target `generous` remains in the dataframe.


In [ ]:
# Drop columns
cols_to_drop = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "payment_type",
    "trip_distance",
    "tip_amount",
    "tip_percent",
    "tolls_amount",
    "total_amount",
    "fare_amount",
    "mta_tax",
    "extra",
    "improvement_surcharge",
    "store_and_fwd_flag",
]

df1 = df1.drop(columns=cols_to_drop)
df1.info()


### Encode categorical variables


In [ ]:
# Convert numeric categorical columns to strings
cols_to_str = [
    "RatecodeID",
    "PULocationID",
    "DOLocationID",
]

for col in cols_to_str:
    df1[col] = df1[col].astype(str)


In [ ]:
# Convert categorical variables to binary
df2 = pd.get_dummies(
    df1,
    drop_first=True,
    dtype=int,
)


### Evaluation metric and class balance


In [ ]:
# Get class balance of generous column
print(df2["generous"].value_counts())
print()
df2["generous"].value_counts(normalize=True)


The target is nearly balanced. Because both false positives and false negatives matter, the primary metric is the **F1 score**.


## PACE: Construct

### Task 3. Modeling


### Split the data


In [ ]:
# Isolate target variable
y = df2["generous"]

# Isolate features
X = df2.drop(columns=["generous"])

# Split into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42,
)


## Random forest


In [ ]:
# Instantiate the random forest classifier
rf = RandomForestClassifier(random_state=42)

# Hyperparameter grid
cv_params = {
    "max_depth": [10, 20],
    "max_features": ["sqrt"],
    "max_samples": [0.7],
    "min_samples_leaf": [1, 2],
    "min_samples_split": [2, 3],
    "n_estimators": [100, 300],
}

# Scoring metrics
scoring = ["accuracy", "precision", "recall", "f1"]

# Grid search
rf1 = GridSearchCV(
    estimator=rf,
    param_grid=cv_params,
    scoring=scoring,
    cv=3,
    refit="f1",
    n_jobs=-1,
)


In [ ]:
# Fit the random forest grid search
rf1.fit(X_train, y_train)


### Optional model persistence


In [ ]:
# Colab-safe directory for saving models
MODEL_PATH = (
    Path("/content")
    if Path("/content").exists()
    else Path.cwd()
)


In [ ]:
def write_pickle(path, model_object, save_name: str):
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)

    with open(path / f"{save_name}.pickle", "wb") as to_write:
        pickle.dump(model_object, to_write)


In [ ]:
def read_pickle(path, saved_model_name: str):
    path = Path(path)

    with open(path / f"{saved_model_name}.pickle", "rb") as to_read:
        model = pickle.load(to_read)

    return model


In [ ]:
# Examine best score
print(
    "Mejor puntaje F1 (validación cruzada): "
    f"{rf1.best_score_:.4f}"
)


In [ ]:
# Examine best parameters
rf1.best_params_


### Results helper


In [ ]:
def make_results(model_name: str, model_object, metric: str):
    # Map the requested metric to the GridSearchCV result column
    metric_dict = {
        "precision": "mean_test_precision",
        "recall": "mean_test_recall",
        "f1": "mean_test_f1",
        "accuracy": "mean_test_accuracy",
    }

    cv_results = pd.DataFrame(model_object.cv_results_)

    best_estimator_results = cv_results.iloc[
        cv_results[metric_dict[metric]].idxmax(),
        :
    ]

    return pd.DataFrame(
        {
            "model": [model_name],
            "precision": [
                best_estimator_results.mean_test_precision
            ],
            "recall": [
                best_estimator_results.mean_test_recall
            ],
            "F1": [
                best_estimator_results.mean_test_f1
            ],
            "accuracy": [
                best_estimator_results.mean_test_accuracy
            ],
        }
    )


In [ ]:
results = make_results("RF1", rf1, "f1")
results


In [ ]:
# Predict on test data
rf_preds = rf1.predict(X_test)


In [ ]:
# Direct test metrics
test_accuracy = accuracy_score(y_test, rf_preds)
test_precision = precision_score(y_test, rf_preds)
test_recall = recall_score(y_test, rf_preds)
test_f1 = f1_score(y_test, rf_preds)

test_scores = pd.DataFrame(
    {
        "Metric": [
            "Accuracy",
            "Precision",
            "Recall",
            "F1 Score",
        ],
        "Score": [
            test_accuracy,
            test_precision,
            test_recall,
            test_f1,
        ],
    }
)

test_scores


In [ ]:
def get_test_scores(model_name: str, preds, y_test_data):
    accuracy = accuracy_score(y_test_data, preds)
    precision = precision_score(y_test_data, preds)
    recall = recall_score(y_test_data, preds)
    f1 = f1_score(y_test_data, preds)

    return pd.DataFrame(
        {
            "model": [model_name],
            "precision": [precision],
            "recall": [recall],
            "F1": [f1],
            "accuracy": [accuracy],
        }
    )


In [ ]:
# Random forest test results
rf_test_scores = get_test_scores(
    "Random Forest",
    rf_preds,
    y_test,
)

rf_test_scores


In [ ]:
# Compare validation and test results
combined_results = pd.concat(
    [results, rf_test_scores],
    axis=0,
).reset_index(drop=True)

combined_results


The validation and test scores should be close if the model generalizes well and is not heavily overfit.


## XGBoost


In [ ]:
# Instantiate XGBoost classifier
xgb = XGBClassifier(
    objective="binary:logistic",
    random_state=42,
    eval_metric="logloss",
)

# Hyperparameter grid
cv_params = {
    "max_depth": [4, 6, 8],
    "min_child_weight": [1, 3, 5],
    "learning_rate": [0.1, 0.2, 0.3],
    "n_estimators": [100, 300],
}

# Scoring metrics
scoring = ["accuracy", "precision", "recall", "f1"]

# Grid search
xgb1 = GridSearchCV(
    estimator=xgb,
    param_grid=cv_params,
    scoring=scoring,
    cv=4,
    refit="f1",
    n_jobs=-1,
)


In [ ]:
# Fit XGBoost grid search
xgb1.fit(X_train, y_train)


In [ ]:
# Examine best score
xgb1.best_score_


In [ ]:
# Examine best parameters
xgb1.best_params_


In [ ]:
# XGBoost validation results
xgb1_results = make_results(
    "XGB1",
    xgb1,
    "f1",
)

xgb1_results


In [ ]:
# Predict on test data
xgb_preds = xgb1.best_estimator_.predict(X_test)


In [ ]:
# XGBoost test results
xgb_test_scores = get_test_scores(
    "XGBoost test",
    xgb_preds,
    y_test,
)

xgb_test_scores


In [ ]:
# Compare validation and test results
xgb_compare = pd.concat(
    [xgb1_results, xgb_test_scores],
    axis=0,
).reset_index(drop=True)

xgb_compare


In [ ]:
# Compare all model results
all_model_results = pd.concat(
    [
        results,
        rf_test_scores,
        xgb1_results,
        xgb_test_scores,
    ],
    axis=0,
).reset_index(drop=True)

all_model_results


The two models can be compared using F1, precision, recall, and accuracy. XGBoost is selected when it provides the best balance and stable validation/test performance.


### Confusion matrix


In [ ]:
# Generate confusion matrix
cm = confusion_matrix(
    y_test,
    xgb_preds,
    labels=xgb1.classes_,
)

# Plot confusion matrix
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=[
        "No generoso",
        "Generoso",
    ],
)

disp.plot(
    values_format="",
    cmap="Blues",
)

plt.title("Matriz de confusión: modelo XGBoost")
plt.show()


Inspect the matrix to determine which error is more common:

- False positive: predicted generous, actually not generous.
- False negative: predicted not generous, actually generous.


### Feature importance


In [ ]:
# Plot the 10 most important XGBoost features by gain
plot_importance(
    xgb1.best_estimator_,
    importance_type="gain",
    max_num_features=10,
)

plt.title("Importancia de variables - XGBoost")
plt.show()


## PACE: Execute

### Task 4. Conclusion

The XGBoost model combines a sequence of decision trees. Each new tree focuses on correcting errors made by earlier trees.

Potential improvements include:

- Interaction between pickup location and time.
- Distance per minute.
- Airport trip indicator.
- Weather conditions.
- Traffic congestion.
- Privacy-preserving historical tipping behavior.
- Vehicle type.

### Ethical recommendation

The model should not be used to deny or delay service to individual passengers. A safer application is aggregate analysis for business planning, driver training, service policy, and understanding broad tipping patterns.


**Congratulations! You have completed the lab.**
